# NB 01 — LatAm ADR Universe: Live Construction & Descriptive Diagnostics

*Stage 1 of a four-notebook pipeline — universe → pairs engine → AI conditioning → portfolio ablation — built for US-listed Latin American ADRs.*

### What it does

A point-in-time strategy requires point-in-time data: conditioning trades on what was known *as of* a past date requires data that can answer "what did this look like back then?" This notebook builds that foundation on **EODHD** — ~11 years of daily prices, full fundamentals, and a complete earnings calendar — and exports a clean, liquidity-screened, survivorship-controlled universe plus the price / return / earnings panels NB 02's OU engine consumes, under stable filenames NB02 consumes directly.

### Reproducibility

Live pulls require an `EODHD_API_KEY` in a local `.env`. The first keyed run pulls, cleans, filters, and writes a snapshot to `data/processed/`; `OFFLINE_MODE=1` then replays the whole pipeline from disk with no network or key. The snapshot is the vendor's data and is **not redistributed**: a fresh clone has an empty `data/processed/`, so reproducing results requires EODHD access to build it first.

---

### Universe definition

LatAm membership is by **primary country of operations**, not listing exchange (always the US) or country of incorporation (often a tax-favourable jurisdiction — Luxembourg, Cayman, BVI). Scope: MSCI LatAm constituents (Brazil, Mexico, Chile, Colombia, Peru, Argentina) plus off-index US-listed ADRs with ≥70% LatAm operations. The authoritative map lives in `data/static/country_of_origin.yaml`; §3 cross-checks it against the vendor's HQ field and flags disagreements.

### Structural caveats

- **Sector concentration.** LatAm equities tilt heavily to cyclicals (Materials, Financials, Consumer Staples); tech is thin vs broad EM, and single-name concentration is high, especially in Brazil.
- **FX-amplified volatility.** ADR returns embed currency exposure; raw vol substantially exceeds underlying business vol.
- **Currency reporting.** Absolute figures (EBITDA, Revenue) are in local accounting currency and are *not* cross-comparable; use ratios (EV/EBITDA, margins, ROE) instead.
- **ADR ratio gotcha.** Some ADRs represent multiple local shares; the vendor occasionally mixes the unit basis — §3 detects and quarantines nonsensical free-float values.

## 1. Preamble — Imports, Paths, Configuration


#### 1.1 Imports & repo bootstrap
Standard imports plus a small bootstrap that adds the repo root to `sys.path` so `from src.config` and `from src.io` resolve regardless of where Jupyter was launched.

In [ ]:
# === Imports & repo bootstrap ===
import sys, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import ticker as mtick, patheffects as pe
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import squareform
import networkx as nx
import requests
import yaml
from IPython.display import display, Markdown

def _add_repo_root_to_syspath():
    cur = Path.cwd().resolve()
    for parent in [cur, *cur.parents]:
        if (parent / "requirements.txt").exists() or (parent / ".git").exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            return
_add_repo_root_to_syspath()

from src.config import PATHS, OFFLINE_MODE, EODHD_API_KEY, ROOT
from src.io import (EODHDClient, flatten_fundamentals,
                    persist_universe_master, persist_panels,
                    persist_earnings, persist_universe_tier,
                    pull_universe_fundamentals, pull_eod_panels,
                    pull_earnings_calendar)

RAW, PROCESSED, STATIC, IMGDIR, CACHE_DIR = PATHS.RAW, PATHS.PROCESSED, PATHS.STATIC, PATHS.IMG, PATHS.CACHE

#### 1.2 Reproducibility & plot styles
Deterministic seed and global seaborn / matplotlib defaults — set once at notebook startup.

In [ ]:
# === Reproducibility & plot styles ===
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 40)

#### 1.3 Configuration
Single source of truth for every snapshot date, universe threshold, and lookback window used downstream. Edit values here, not in §3-§7.

In [ ]:
# === Configuration (single source of truth for all magic numbers) ============
CONFIG = {
    # --- Snapshot & sample windows ---
    "SNAPSHOT_DATE":         pd.Timestamp("2025-03-31"),
    "SAMPLE_YEARS":          10,
    "SAMPLE_START":          pd.Timestamp("2015-01-01"),   # EOD history start (lookback before backtest)

    # --- Universe thresholds (strict / loose) ---
    "MCAP_MIN_STRICT":       2e9,
    "ADV_MIN_STRICT":        200_000,
    "DOLLAR_MIN_STRICT":     5e6,
    "MCAP_MIN_LOOSE":        1e8,
    "ADV_MIN_LOOSE":         50_000,
    "DOLLAR_MIN_LOOSE":      1e6,
    "PAIRS_COVERAGE_DAYS":   756,
    "PAIRS_COVERAGE_MIN":    0.60,

    # --- Correlation diagnostics ---
    "CORR_LOOKBACK_DAYS":    756,
    "CORR_THRESHOLD":        0.70,
}

#### 1.4 Environment audit
Stack versions, key paths, and OFFLINE vs. live status — printed once as an audit trail.

In [ ]:
# === Version stamping (audit trail) ===
print(f"[INFO] python={sys.version.split()[0]} pandas={pd.__version__} "
      f"numpy={np.__version__} scipy={__import__('scipy').__version__} "
      f"networkx={nx.__version__} requests={requests.__version__}")
print(f"[INFO] repo      = {ROOT.name}")
print(f"[INFO] RAW       = {RAW.relative_to(ROOT)}")
print(f"[INFO] CACHE_DIR = {CACHE_DIR.relative_to(ROOT)}")
if OFFLINE_MODE:
    print("[INFO] OFFLINE_MODE — replaying from local data/processed/ snapshot (no EODHD calls)")
    if not EODHD_API_KEY:
        print("[INFO] (no EODHD_API_KEY present — offline is the only option)")
else:
    print(f"[OK]   EODHD_API_KEY loaded (len={len(EODHD_API_KEY)}) — live pull enabled")

## 2. EODHD Client with Disk Cache

Thin wrapper around the EODHD All-In-One REST API with an on-disk JSON cache.

**Cache design**

- Key: `SHA256(endpoint + "?" + sorted-params-without-api_token)`
- Layout: `data/raw/eodhd_cache/<sha256>.json` — flat, hash-named, no sidecar
- TTL: per-endpoint via `CONFIG["EODHD_TTL"]`, longest-prefix match. TTL is a
  **re-pull cadence**, not a staleness threshold — pinning of exact payloads
  for reproducibility happens via `data/manifest.json` (sha256 + row_count +
  snapshot_date per artifact), set elsewhere.
- The API token is never part of the cache key, never written to disk, never
  logged. Cache directory is therefore portable across machines/keys.
- Writes are atomic (`tmp → rename`) so an interrupted fetch cannot poison
  the cache.

**Public surface** (matches the pull contract):

- `fundamentals(code)` — General slim, Gic*, AddressData, HomeCategory,
  IsDelisted, IPODate, Highlights, Valuation, SharesStats
- `eod(code, from_date, to_date)` — full OHLCV; caller coerces numerics
- `earnings_calendar(from_date, to_date, symbols)` — code, report_date, date,
  before_after_market, currency, actual, estimate, percent
- `news(code, from_date, to_date, limit)` — EODHD news for Layer 2 attribution


In [ ]:
# === EODHD client (cache-backed; implementation lives in src/io.py) ===
client = EODHDClient()

### 2a. Smoke test — MELI.US fundamentals

Two back-to-back calls. The first exercises whichever path is appropriate
(cold → network → write, or warm → hit if the cache survives from a prior
session); the second must be a cache hit and return a byte-identical payload.

Then we introspect EODHD's actual schema (top-level keys + `General` keys)
and check that the pull contract holds. `AddressData` is verified at its
real location — nested under `General` — and the HQ country is taken from
`General.AddressData.Country`, while `General.CountryName` (the listing
country) is printed alongside it so any HQ-vs-listing disagreement is
visible. Both fields will inform the contamination check in §3.


In [ ]:
# === Smoke test: MELI.US fundamentals ===
if OFFLINE_MODE:
    print("[OFFLINE] skipping live smoke test")
else:
    # --- First call (cache miss on a fresh tree; hit if cache survived) ---
    t0 = time.perf_counter()
    meli = client.fundamentals("MELI.US")
    t_first = (time.perf_counter() - t0) * 1000

    # --- Second call: must be a cache hit ---
    t0 = time.perf_counter()
    meli2 = client.fundamentals("MELI.US")
    t_second = (time.perf_counter() - t0) * 1000

    print(f"[1st] {t_first:7.1f} ms   {client.stats}")
    print(f"[2nd] {t_second:7.1f} ms   {client.stats}")
    assert meli == meli2,             "cached payload must be byte-identical to source"
    assert client.stats["hits"] >= 1, "second call should be a cache hit"
    # Timing inequality is only meaningful if the 1st call actually hit the network.
    if client.stats["misses"] >= 1:
        assert t_second < t_first, (f"after a real cache miss, the 2nd (hit) call should be cheaper; "
                                    f"got 1st={t_first:.1f}ms 2nd={t_second:.1f}ms")


    # --- Schema introspection (committed to notebook output as an audit trail) ---
    print(f"\n[schema] top-level blocks: {sorted(meli.keys())}")
    print(f"[schema] General keys     : {sorted(meli['General'].keys())}")
    has_addr = "AddressData" in meli["General"]
    print(f"[schema] General.AddressData present: {has_addr}")
    if has_addr:
        print(f"[schema] AddressData keys : {sorted(meli['General']['AddressData'].keys())}")


    # --- Field presence per v2 pull contract -------------------------------------
    # Contract (locked):
    #   General        : Code, Name, Exchange, IsDelisted, IPODate, HomeCategory,
    #                    GicSector, GicGroup, GicIndustry, GicSubIndustry, AddressData
    #   General.AddressData.Country  (HQ country — primary classifier)
    #   Highlights     : MarketCapitalization (+ MarketCapitalizationMln)
    #   Valuation      : (full block)
    #   SharesStats    : SharesFloat, SharesOutstanding  → FreeFloatPercent
    # -----------------------------------------------------------------------------
    required = {
        "General":     ["Code", "Name", "Exchange", "IsDelisted", "IPODate",
                        "HomeCategory", "GicSector", "GicGroup",
                        "GicIndustry", "GicSubIndustry", "AddressData"],
        "Highlights":  ["MarketCapitalization"],
        "Valuation":   ["TrailingPE", "EnterpriseValueEbitda"],
        "SharesStats": ["SharesFloat", "SharesOutstanding"],
    }
    missing = []
    for block, keys in required.items():
        if block not in meli:
            missing.append(block); continue
        for k in keys:
            if k not in meli[block]:
                missing.append(f"{block}.{k}")
    # Nested check: General.AddressData.Country
    if "AddressData" in meli.get("General", {}):
        if "Country" not in meli["General"]["AddressData"]:
            missing.append("General.AddressData.Country")
    assert not missing, f"v2 contract gaps: {missing}"


    # --- HQ vs listing country (the two flavours of 'Country' EODHD exposes) -----
    g   = meli["General"]
    hq  = g["AddressData"].get("Country")            # HQ — primary classifier
    lst = g.get("CountryName") or g.get("CountryISO") # listing country
    agree = "✓ same" if hq == lst else "≠ disagree (HQ ≠ listing)"
    print(f"\n[country] AddressData.Country (HQ)   = {hq!r}")
    print(f"[country] CountryName       (listing) = {lst!r}   {agree}")


    # --- FreeFloatPercent derivation per locked contract -------------------------
    ss = meli["SharesStats"]
    sf, so = ss.get("SharesFloat"), ss.get("SharesOutstanding")
    if sf and so:
        fff = sf / so * 100
        print(f"\n[OK]   FreeFloatPercent = {fff:.2f}%   "
              f"(float={sf:,.0f}  out={so:,.0f})")
    else:
        print(f"\n[WARN] SharesFloat={sf}  SharesOutstanding={so} "
              f"— FreeFloatPercent unavailable")


    # --- Identity spot-check -----------------------------------------------------
    print(f"\n[OK]   {g['Code']} | {g['Name']} | {g['Exchange']} | "
          f"HomeCategory={g.get('HomeCategory')} | IPODate={g.get('IPODate')} | "
          f"IsDelisted={g.get('IsDelisted')}")


    # --- Sanity: force=True bypasses cache and increments 'forced' ---------------
    forced_before = client.stats["forced"]
    _ = client.fundamentals("MELI.US", force=True)
    assert client.stats["forced"] == forced_before + 1, "force=True must bypass cache"
    print(f"[OK]   force=True path exercised   {client.stats}")

## 3. Universe Master Table — Load, Pull, Validate

Assembles the master fundamentals DataFrame for the 78 active LatAm ADRs by
loading the static authority files (`seed_tickers.yaml`, `country_of_origin.yaml`),
fetching EODHD fundamentals for each ticker through the cached client, flattening
each JSON payload into a single tidy row, and persisting the result with a
manifest entry.

**Column scope** (narrow, per locked decision): everything in `adr_fundamentals.csv`
(so NB1 §4-§7 and any downstream consumer keeps working)
plus the new enrichment columns (`Gic*`, `IPODate`, `HomeCategory`, `IsDelisted`,
`HQCountryEODHD`, derived `FreeFloatPercent`). Richer EODHD blocks
(`Earnings`, `Financials`, `AnalystRatings`, `Holders`, etc.) stay in their
JSON form in `data/raw/eodhd_cache/` for NB3 to consume on demand. `ADV_63` and
`DollarVolume_3M` depend on the EOD panel and are joined in §4, not here.

**Failure posture**: per-ticker quality flags, no abort on partial gaps. A
ticker with `Highlights.EBITDA = None` (common for early-stage names like NU)
still enters the universe; the gap is logged and surfaced in the §3.2 summary.
The loop only fails a ticker on a hard error (HTTP 4xx/5xx, network outage).

### 3.0 Load static authority


In [ ]:
# === 3.0 Load yamls + validate consistency ===

# Active list + excluded reasons
seeds   = yaml.safe_load((STATIC / "seed_tickers.yaml").read_text())
ACTIVE   = list(seeds["active"])
EXCLUDED = dict(seeds["excluded"])

# Country of operations (inverted: country -> list[ticker]) → ticker -> country
groups = yaml.safe_load((STATIC / "country_of_origin.yaml").read_text())
ticker_to_country = {t: c for c, ts in groups.items() for t in ts}

# --- Consistency assertions (catch yaml drift loudly) ---
# Every active ticker has a CountryOfOrigin entry
missing_co = [t for t in ACTIVE if t not in ticker_to_country]
assert not missing_co, f"active tickers missing country_of_origin: {missing_co}"

# No orphans: every ticker in country_of_origin is either active or excluded
known = set(ACTIVE) | set(EXCLUDED)
orphans = sorted(t for t in ticker_to_country if t not in known)
assert not orphans, f"country_of_origin orphans (neither active nor excluded): {orphans}"

# No collisions between active and excluded
overlap = set(ACTIVE) & set(EXCLUDED)
assert not overlap, f"tickers in both active and excluded: {overlap}"

# Per-country counts
from collections import Counter
country_counts = Counter(ticker_to_country[t] for t in ACTIVE)
print(f"[OK]   {len(ACTIVE)} active, {len(EXCLUDED)} excluded, "
      f"{len(country_counts)} countries")
for c in sorted(country_counts, key=lambda x: -country_counts[x]):
    print(f"       {country_counts[c]:3d} × {c}")


### 3.1 Pre-flight: flatten three reference tickers

`flatten_fundamentals` takes a raw EODHD fundamentals payload and produces
`(row_dict, quality_flags)`. The schema is fixed and ordered: `Ticker`,
General slim, `HQCountryEODHD`, `CountryOfOrigin`, full `Highlights`, full
`Valuation`, `SharesStats` subset + derived `FreeFloatPercent`. `None` values
are tolerated and recorded as quality flags rather than treated as errors.

Pre-flight runs the flattener on **MELI / SUPV / BAP** — the same triplet that
was schema-inventoried in `data/raw/eodhd_inventory/`. A large well-covered
name, a thin-coverage Argentine bank, a Peruvian financial — covers the
practical range of fundamentals completeness before the full loop.


In [ ]:
# === 3.1 Pre-flight — imported flatten_fundamentals on three reference tickers ===

# --- Pre-flight on 3 reference tickers ---
if OFFLINE_MODE:
    print("[OFFLINE] skipping preflight (live fetch)")
else:

    PREFLIGHT = ["MELI", "SUPV", "BAP"]
    print(f"[preflight] flattener on {PREFLIGHT}\n")
    for t in PREFLIGHT:
        payload = client.fundamentals(f"{t}.US")
        row, flags = flatten_fundamentals(t, payload, ticker_to_country.get(t))
        mc = row["MarketCapitalization"]
        print(f"  --- {t} ---")
        print(f"    Name        : {row['Name']}")
        print(f"    Sector/Ind  : {row['Sector']} / {row['Industry']}")
        print(f"    GicSector   : {row['GicSector']}")
        print(f"    HQ (vendor) : {row['HQCountryEODHD']!r:12s}  CountryOfOrigin: {row['CountryOfOrigin']!r}")
        print(f"    MarketCap   : ${mc/1e9:.2f}B" if mc else "    MarketCap   : None")
        print(f"    FreeFloat   : {row['FreeFloatPercent']:.2f}%" if row['FreeFloatPercent'] else "    FreeFloat   : None")
        print(f"    TrailingPE  : {row['TrailingPE']}    EV/EBITDA: {row['EnterpriseValueEbitda']}")
        print(f"    Quality     : {'clean' if not flags else f'{len(flags)} gap(s) — first 3: ' + str(flags[:3])}")
        print()
    print(f"[OK]   cache stats after pre-flight: {client.stats}")

### 3.2 Full pull across the active universe

Iterates over `ACTIVE`, fetches fundamentals through the cached client (so all
but the first run is dominated by ~few-ms cache hits per ticker), flattens
each payload, and assembles the master `universe` DataFrame indexed by Ticker.

Quality flags are aggregated by (a) frequency across tickers — which gaps are
systemic vs idiosyncratic — and (b) worst-covered tickers, so the candidates
for `excluded.yaml` promotion are obvious.

Hard errors (HTTP 4xx/5xx, network) fail just that ticker and are reported at
the end; the loop completes regardless.


In [ ]:
# === 3.2 Full pull + master DataFrame ===
from collections import Counter
snapshot = CONFIG["SNAPSHOT_DATE"].strftime("%Y-%m-%d")

universe, quality, errors = pull_universe_fundamentals(
    client, ACTIVE, ticker_to_country,
    snapshot=snapshot, offline=OFFLINE_MODE, processed_dir=PROCESSED,
)
print(f"[OK]   universe shape : {universe.shape}")
if not OFFLINE_MODE:
    print(f"[OK]   cache stats     : {client.stats}")
print(f"[OK]   failed tickers  : {len(errors)}")
for t, msg in errors.items():
    print(f"        {t}: {msg}")

# --- Quality summary (narrative; stays in the notebook) ---
print(f"\n[quality] tickers with at least one gap: {len(quality)} / {len(universe)}")
if quality:
    flag_freq = Counter(f for flags in quality.values() for f in flags)
    print(f"\n[quality] gap frequency (top 12, count of tickers affected):")
    for flag, n in flag_freq.most_common(12):
        print(f"          {n:3d} × {flag}")
    worst = sorted(quality.items(), key=lambda x: -len(x[1]))[:5]
    print(f"\n[quality] worst-covered tickers (top 5 by gap count):")
    for t, flags in worst:
        print(f"          {t} ({len(flags)} gaps): {flags[:3]}{'...' if len(flags) > 3 else ''}")

### 3.3 HQ vs operations cross-check (categorised)

Most LatAm ADRs are domiciled in tax-favourable jurisdictions (Luxembourg,
Cayman Islands, Bermuda, BVI) or in the US/Canada, while operating in LatAm.
That's a structural feature of the ADR market — *not* a finding worth a
warning every time. The real audit concern is **regional shifts** —
tickers whose vendor HQ is *in LatAm but a different country* than the
operations country (e.g., MELI: Uruguay HQ but Argentina-rooted operations).

Disagreements are categorised into:

- `agree` — vendor HQ == yaml CountryOfOrigin (the bulk of the universe).
- `tax-haven` — HQ in LUX/KY/VG/BMU/IE/etc. Common ADR structure;
  informational.
- `non-LatAm` — HQ in US/CA/GB/etc. Common for resource extractors and
  US-incorporated multinationals (SCCO, TGLS, PSMT, SGML, PAAS);
  informational.
- `regional-shift` — HQ in LatAm but != CountryOfOrigin. **This is the
  category that needs review.** WARNs fire only on shifts not pre-flagged
  in `country_of_origin.yaml`'s header docstring.
- `vendor-missing` — HQ field absent in payload.

The pre-flagged set is `{ARCO, MELI, VIST, PAAS}`, documented in the yaml
header. The three regional shifts (ARCO Uruguay→AR, MELI Uruguay→AR,
VIST Mexico→AR) are intentional operations-not-HQ classifications; PAAS
is a non-LatAm domicile (Canada→Pan-Regional). Any future ticker that
shows up under `regional-shift` outside this set will trigger a WARN
and warrants either a yaml update or a vendor-data review.


In [ ]:
# === 3.3 HQ vs CountryOfOrigin cross-check (categorised) ===

TAX_HAVENS  = {"Luxembourg", "Cayman Islands", "British Virgin Islands",
               "Bermuda", "Ireland", "Jersey", "Guernsey", "Isle Of Man"}
LATAM       = {"Argentina", "Brazil", "Chile", "Colombia", "Peru",
               "Mexico", "Uruguay", "Panama", "Venezuela",
               "Ecuador", "Bolivia", "Paraguay"}
PRE_FLAGGED = {"ARCO", "MELI", "VIST", "PAAS"}   # documented in country_of_origin.yaml header

def categorize_hq(hq: str | None, ops: str) -> str:
    if hq is None:           return "vendor-missing"
    if hq == ops:            return "agree"
    if hq in TAX_HAVENS:     return "tax-haven"
    if hq in LATAM:          return "regional-shift"
    return "non-LatAm"

universe["HQCategory"] = [
    categorize_hq(hq, ops)
    for hq, ops in zip(universe["HQCountryEODHD"], universe["CountryOfOrigin"])
]

disagree = universe.loc[universe["HQCountryEODHD"] != universe["CountryOfOrigin"]]
print(f"[cross-check] {len(disagree)} of {len(universe)} tickers disagree "
      f"(vendor HQ ≠ yaml CountryOfOrigin)\n")
print(f"[cross-check] category breakdown:")
for cat, n in disagree["HQCategory"].value_counts().items():
    print(f"              {n:3d} × {cat}")

view = (disagree[["Name", "HQCountryEODHD", "CountryOfOrigin", "HQCategory"]]
        .sort_values(["HQCategory", "CountryOfOrigin", "HQCountryEODHD"]))
display(view)

# --- Surface only audit-relevant cases ---
regional_shifts     = set(disagree.loc[disagree["HQCategory"] == "regional-shift"].index)
unexpected_shifts   = sorted(regional_shifts - PRE_FLAGGED)
missing_pre_flagged = sorted(PRE_FLAGGED - set(disagree.index))

print()
if unexpected_shifts:
    print(f"[WARN]   unexpected regional shift(s) (LatAm-to-LatAm, not pre-flagged): {unexpected_shifts}")
if missing_pre_flagged:
    print(f"[WARN]   pre-flagged ticker(s) no longer disagree (vendor data changed?): {missing_pre_flagged}")
if not unexpected_shifts and not missing_pre_flagged:
    in_pf_regional = sorted(regional_shifts & PRE_FLAGGED)
    print(f"[OK]     regional shifts all documented in country_of_origin.yaml: {in_pf_regional or 'none observed'}")
    print(f"[OK]     tax-haven / non-LatAm HQ are informational metadata, not warnings") 

### 3.4 Persist universe + manifest entry

Writes `data/processed/fundamentals_<snapshot>.csv` and records the artifact
in `data/manifest.json` per the locked spec — `sha256`, `row_count`,
`snapshot_date`. `column_count` and a UTC `generated_at` go in alongside as
defensible extras for audit.

The manifest is read-modify-written: existing entries for other artifacts are
preserved. Re-running this cell overwrites the row for this artifact path
only.


In [ ]:
# === 3.4 Persist + manifest ===
snapshot  = CONFIG["SNAPSHOT_DATE"].strftime("%Y-%m-%d")
written   = persist_universe_master(universe, snapshot)
fund_path = written["fundamentals"]

print(f"[OK]   wrote {fund_path.relative_to(ROOT)}")
print(f"       bytes  : {fund_path.stat().st_size:,}")
print(f"       rows   : {len(universe)}")
print(f"       cols   : {universe.shape[1]}")

## 4. EOD Price Panel — Live Pull, Coverage, Liquidity

Live EODHD OHLCV history for the 78 active tickers, from `CONFIG["SAMPLE_START"]`
(2015-01-01) up to the latest available close — ~11 years of history that refreshes
to yesterday's close on every run (EOD cache TTL = 1 day).

Builds three wide panels (date × ticker):

- `prices` — **adjusted close** (total-return adjusted for splits/dividends;
  the correct series for spread construction in NB02's OU model).
- `volume` — raw share volume.
- `returns` — simple daily returns (`prices.pct_change()`), feeds the
  correlation diagnostics in §6.

Then derives the two liquidity columns (`ADV_63`, `DollarVolume_3M`), computed
point-in-time over the 63 trading days ending at `SNAPSHOT_DATE`, and joins them
back onto the `universe` table. Panels persist as parquet; the universe CSV is
re-written with the liquidity columns; manifest entries record date spans + sha256.

### 4.1 Pull EOD price / volume / returns panels


In [ ]:
# === 4.1 EOD pull + panel assembly ===
snapshot     = CONFIG["SNAPSHOT_DATE"].strftime("%Y-%m-%d")
SAMPLE_START = CONFIG["SAMPLE_START"].strftime("%Y-%m-%d")

prices, volume, returns, eod_errors = pull_eod_panels(
    client, ACTIVE, from_date=SAMPLE_START,
    snapshot=snapshot, offline=OFFLINE_MODE, processed_dir=PROCESSED,
)
print(f"[OK]   fetched EOD for {len(prices.columns)}/{len(ACTIVE)} tickers")
if not OFFLINE_MODE:
    print(f"[OK]   cache stats: {client.stats}")
if eod_errors:
    print(f"[WARN] {len(eod_errors)} failure(s):")
    for t, msg in eod_errors.items():
        print(f"        {t}: {msg}")
print(f"[OK]   prices  panel: {prices.shape}  ({prices.index.min().date()} → {prices.index.max().date()})")
print(f"[OK]   volume  panel: {volume.shape}")
print(f"[OK]   returns panel: {returns.shape}")

### 4.2 Coverage diagnostics

Tickers IPO at different times, so the panel is ragged on the left (recent
listings like NU, INTR, LVRO, AUNA start late). This table reports each
ticker's span and its coverage over the trailing `PAIRS_COVERAGE_DAYS`
(756 ≈ 3 years) window the pairs filter uses. Thin names
(< `PAIRS_COVERAGE_MIN` = 60% recent coverage) are surfaced — they're not
dropped here (that happens in §6), just flagged.


In [ ]:
# === 4.2 Coverage diagnostics ===
cov = pd.DataFrame({
    "first_date": prices.apply(lambda c: c.first_valid_index()),
    "last_date":  prices.apply(lambda c: c.last_valid_index()),
    "n_obs":      prices.notna().sum(),
})
cov["years"] = (cov["last_date"] - cov["first_date"]).dt.days / 365.25

win_days = CONFIG["PAIRS_COVERAGE_DAYS"]
recent   = prices.tail(win_days)
cov["coverage_recent"] = (recent.notna().sum() / len(recent)).round(3)
cov = cov.sort_values("coverage_recent")

print(f"[coverage] panel spans {prices.index.min().date()} → {prices.index.max().date()} "
      f"({len(prices)} trading days)")
print(f"[coverage] median history: {cov['years'].median():.1f} yrs   "
      f"min: {cov['years'].min():.1f}   max: {cov['years'].max():.1f}")

thin = cov[cov["coverage_recent"] < CONFIG["PAIRS_COVERAGE_MIN"]]
print(f"\n[coverage] {len(thin)} thin name(s) (recent {win_days}d coverage < "
      f"{CONFIG['PAIRS_COVERAGE_MIN']:.0%}):")
display(thin[["first_date", "last_date", "n_obs", "years", "coverage_recent"]])

# --- Delisting / survivorship ledger ---
delist_cut = CONFIG["SNAPSHOT_DATE"] - pd.Timedelta(days=30)
delisted   = cov[cov["last_date"] < delist_cut].copy()
if len(delisted):
    print(f"\n[survivorship] {len(delisted)} ticker(s) stopped trading >30d before "
          f"snapshot ({CONFIG['SNAPSHOT_DATE'].date()}) — delisted/halted in-sample:")
    display(delisted[["first_date", "last_date", "n_obs", "years"]])
    print("[survivorship] dropped at the §7 coverage filter; logged here for audit "
          "(survivorship-bias control)")
else:
    print(f"\n[survivorship] no tickers delisted before snapshot — clean panel")

### 4.3 Liquidity metrics (ADV, dollar volume)

`ADV_63` (63-day average daily share volume) and `DollarVolume_3M`
(63-day average of close × volume) are computed point-in-time over the 63
trading days **ending at `SNAPSHOT_DATE`** — liquidity as judged when the
universe is formed, not look-ahead from today's tape. Both join back onto
`universe`. Panels are written as parquet; the universe CSV is re-written
with the two new columns; manifest entries record sha256 + date spans.


In [ ]:
# === 4.3 Liquidity metrics + persist ===

# Point-in-time 63-day window ending at (or just before) SNAPSHOT_DATE
asof   = prices.index[prices.index <= CONFIG["SNAPSHOT_DATE"]].max()
WIN    = 63
vol_w  = volume.loc[:asof].tail(WIN)
px_w   = prices.loc[:asof].tail(WIN)

universe["ADV_63"]          = vol_w.mean()
universe["DollarVolume_3M"] = (px_w * vol_w).mean()

print(f"[liquidity] ADV_63 / DollarVolume_3M over {WIN}d ending {asof.date()}")
print(universe[["Name", "ADV_63", "DollarVolume_3M"]]
      .sort_values("DollarVolume_3M", ascending=False).head(10).to_string())

snapshot = CONFIG["SNAPSHOT_DATE"].strftime("%Y-%m-%d")
written  = persist_panels(prices, volume, returns, universe, snapshot)
for name, path in written.items():
    print(f"[OK]   wrote {path.relative_to(ROOT)}  ({path.stat().st_size:,} bytes)")

## 5. Earnings Calendar — Live Pull

Pulls each ticker's earnings calendar from EODHD (`calendar/earnings`) from
`CONFIG["SAMPLE_START"]` forward — historical reported earnings plus any
upcoming scheduled dates.

**Why this matters for the strategy.** Layer 1 (OU mean-reversion) assumes
spread deviations are *temporary* and revert. An earnings announcement can
break that assumption outright: a big beat/miss may permanently re-rate one
leg of a pair (a structural shock, not a transient wiggle). So NB02 gates
trades around these dates — don't open or hold a pair across an earnings
print. This table is the gating input.

Fields (locked contract): `report_date` (announcement), `period_date`
(fiscal quarter end), `before_after_market`, `currency`, `eps_actual`,
`eps_estimate`, `surprise_pct`. Stored as a tidy long table (one row per
ticker-announcement), persisted as parquet.


### 5.1 Pull earnings calendar


In [ ]:
# === 5.1 Earnings calendar pull + assembly ===
snapshot     = CONFIG["SNAPSHOT_DATE"].strftime("%Y-%m-%d")
SAMPLE_START = CONFIG["SAMPLE_START"].strftime("%Y-%m-%d")

earnings, earn_counts, earn_errors = pull_earnings_calendar(
    client, ACTIVE, from_date=SAMPLE_START,
    snapshot=snapshot, offline=OFFLINE_MODE, processed_dir=PROCESSED,
)
print(f"\n[OK]   {len(earnings)} earnings events across "
      f"{earnings['Ticker'].nunique()}/{len(ACTIVE)} tickers")
if not OFFLINE_MODE:
    print(f"[OK]   cache stats: {client.stats}")
if len(earnings) and earnings["report_date"].notna().any():
    print(f"[OK]   report_date span: {earnings['report_date'].min().date()} "
          f"→ {earnings['report_date'].max().date()}")
if earn_errors:
    print(f"[WARN] {len(earn_errors)} failure(s): {earn_errors}")

### 5.2 Coverage diagnostics


In [ ]:
# === 5.2 Earnings coverage + persist ===
ev_per = pd.Series(earn_counts).sort_values()
zero   = ev_per[ev_per == 0]
print(f"[earnings] events/ticker — median {ev_per.median():.0f}  "
      f"min {ev_per.min()}  max {ev_per.max()}")
if len(zero):
    print(f"[WARN]   {len(zero)} ticker(s) with zero events: {list(zero.index)}")

# Historical vs upcoming split
today = pd.Timestamp.today().normalize()
hist  = int((earnings["report_date"] <= today).sum())
fut   = int((earnings["report_date"] >  today).sum())
print(f"[earnings] {hist} historical, {fut} upcoming/scheduled")

# Surprise-data availability (proxy for analyst coverage depth)
has_surp = int(earnings["surprise_pct"].notna().sum())
if len(earnings):
    print(f"[earnings] {has_surp}/{len(earnings)} events carry a surprise % "
          f"({has_surp/len(earnings):.0%}) — thin-coverage names will be lower")

# Thinnest-covered tickers (fewest events) — candidates to watch for gating gaps
print(f"\n[earnings] fewest events (bottom 8):")
for t, n in ev_per.head(8).items():
    print(f"           {t}: {n}")

# --- Persist ---
snapshot  = CONFIG["SNAPSHOT_DATE"].strftime("%Y-%m-%d")
written   = persist_earnings(earnings, snapshot)
earn_path = written["earnings"]
print(f"\n[OK]   wrote {earn_path.relative_to(ROOT)}  "
      f"({earn_path.stat().st_size:,} bytes, {earnings.shape})")

## 6. Universe Diagnostics — Filters, Composition, Correlation

Descriptive layer over the live universe: liquidity/size screen, liquidity-vs-marketcap bubble, sector/industry/country composition, performance snapshot, clustered correlation heatmap, correlation network, and dendrogram. Figures save to `notebooks/img/`.

Note: `universe` here is the full 78-row fundamentals table; the strict-screen output is `strict_tickers` / `funda_liq`.

### 6.1 Liquidity & size filter (strict / institutional tier)


In [ ]:
# === 6.1 Liquidity & size filtering (strict tier) ===

def show_and_save(fig, filename=None):
    """Display a figure and optionally save to notebooks/img/."""
    display(fig)
    if filename:
        fig.savefig(IMGDIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)

# Aliases so the ported logic reads cleanly
fundamentals = universe                # full 78-row table from §3-§4
EXCLUDE      = set(EXCLUDED)            # excluded tickers from seed_tickers.yaml

MCAP_MIN   = CONFIG["MCAP_MIN_STRICT"]
ADV_MIN    = CONFIG["ADV_MIN_STRICT"]
DOLLAR_MIN = CONFIG["DOLLAR_MIN_STRICT"]
REQ        = ["MarketCapitalization", "ADV_63", "DollarVolume_3M"]
fundamentals[REQ] = fundamentals[REQ].apply(pd.to_numeric, errors="coerce")

mask = (
    (fundamentals["MarketCapitalization"] > MCAP_MIN) &
    ((fundamentals["ADV_63"]          > ADV_MIN) |
     (fundamentals["DollarVolume_3M"] > DOLLAR_MIN))
)
funda_liq = fundamentals[mask].copy()
print(f"[INFO] strict screen: {len(fundamentals)} → {len(funda_liq)} tickers")

# Require both price & volume coverage in the live panels
available     = set(prices.columns) & set(volume.columns)
dropped       = sorted(set(funda_liq.index) - available)
if dropped:
    print(f"[WARN] {len(dropped)} filtered tickers lack panel data: {dropped}")
strict_tickers = sorted(set(funda_liq.index) & available)
funda_liq      = funda_liq.loc[strict_tickers]
print(f"[INFO] final strict universe: {len(strict_tickers)} tickers")

print("\n[INFO] strict universe by country of origin:")
for c, n in funda_liq["CountryOfOrigin"].value_counts().items():
    print(f"         {c:18} {n:>3}")

preview = (funda_liq.loc[strict_tickers, REQ + ["CountryOfOrigin", "Sector"]]
           .sort_values("MarketCapitalization", ascending=False).head(10))
display(preview.style.format({"MarketCapitalization":"{:,.0f}",
                              "ADV_63":"{:,.0f}", "DollarVolume_3M":"{:,.0f}"}))

# --- Borrow-risk flag (pairs trading shorts one leg — borrow availability matters) ---
for col in ("FreeFloatPercent", "PercentInsiders", "PercentInstitutions", "ShortPercentFloat"):
    if col not in funda_liq.columns:
        funda_liq[col] = np.nan
funda_liq["BorrowRisk"] = (
    (funda_liq["FreeFloatPercent"].fillna(100) < 30) |
    (funda_liq["PercentInsiders"].fillna(0)    > 50)
)
nrisk = int(funda_liq["BorrowRisk"].sum())
print(f"\n[borrow] {nrisk} strict-universe name(s) flagged borrow-risk "
      f"(free float < 30% or insiders > 50%):")
if nrisk:
    display(funda_liq[funda_liq["BorrowRisk"]]
            [["Name", "FreeFloatPercent", "PercentInsiders", "CountryOfOrigin"]])


### 6.2 Liquidity-adjusted investability profile


In [ ]:
# === 6.2 Liquidity vs Market Cap (bubble) ===
fig, ax = plt.subplots(figsize=(16, 10))
dfp = funda_liq.loc[strict_tickers, ["ADV_63", "MarketCapitalization", "DollarVolume_3M"]].astype(float)
x, y, dv = dfp["ADV_63"]/1e6, dfp["MarketCapitalization"]/1e9, dfp["DollarVolume_3M"]

dv5, dv95 = np.nanpercentile(dv, [5, 95])
dv_clip   = dv.clip(dv5, dv95)
sizes     = 200 + 1200*(dv_clip - dv5)/(dv95 - dv5 + 1e-9)

ax.scatter(x, y, s=sizes, alpha=0.65, color="steelblue", edgecolors="k", linewidth=0.5)
for t in strict_tickers:
    ax.text(x[t]*1.02, y[t]*1.01, t, fontsize=8.5, weight="bold", color="#222",
            ha="left", va="bottom",
            path_effects=[pe.withStroke(linewidth=1.6, foreground="white")])
ax.axvline(ADV_MIN/1e6, color="#999", ls="--", lw=1, alpha=0.8)
ax.axhline(MCAP_MIN/1e9, color="#999", ls="--", lw=1, alpha=0.8)
ax.set_xscale("log")
ax.set_xticks([0.1, 0.3, 1, 3, 10, 30, 60])
ax.set_xticklabels(["0.1","0.3","1","3","10","30","60"], fontweight="bold")
ax.set_xlabel("ADV (million shares/day)", fontweight="bold")
ax.set_ylabel("Market Cap (bn USD)", fontweight="bold")
ax.set_title(f"Figure 1: Liquidity vs Market Cap — Strict Universe ({len(strict_tickers)} ADRs)",
             fontsize=16, fontweight="bold")
ax.grid(True, alpha=0.3, ls="--", which="both")
for ref in (2e8, 5e7, 1e7):
    s = 200 + 1200*((np.clip(ref, dv5, dv95) - dv5)/(dv95 - dv5 + 1e-9))
    ax.scatter([], [], s=s, color="steelblue", edgecolors="k", alpha=0.65, label=f"${ref/1e6:,.0f}M")
lg = ax.legend(loc="lower right", title="3M Dollar Volume", frameon=True)
lg.get_frame().set_alpha(0.9); lg.set_title("3M Dollar Volume", prop={"weight":"bold"})
for t in lg.get_texts(): t.set_fontweight("bold")
show_and_save(fig, "liquidity_vs_marketcap.png")

### 6.3 Composition by sector, industry & country


In [ ]:
# === 6.3 Sector / Industry / Country composition ===
df = funda_liq.loc[strict_tickers]
N  = len(strict_tickers)

fig, axd = plt.subplot_mosaic(
    [["country", "sector"],
     ["industry", "industry"]],
    figsize=(13, 12),
)
fig.subplots_adjust(hspace=0.10, wspace=0.05, top=0.90)

specs = {
    "country":  ("CountryOfOrigin", f"Country of Origin ({N} ADRs)", "Set2",  None),
    "sector":   ("Sector",          f"Sector Mix ({N} ADRs)",        "tab20", None),
    "industry": ("Industry",        "Top-10 Industries",             "tab10", 10),
}
for key, (col, title, cmap, n) in specs.items():
    ax = axd[key]
    vals = df[col].value_counts().head(n) if n else df[col].value_counts()
    if col == "Sector":
        small = vals[vals / vals.sum() < 0.03]
        if len(small) > 1:
            vals = vals.drop(small.index); vals["Other"] = small.sum()
    vals.plot.pie(ax=ax, autopct="%1.0f%%", radius=0.70,
                  colors=plt.get_cmap(cmap)(range(len(vals))),
                  labeldistance=1.18, pctdistance=0.78)
    ax.set_title(title, fontweight="bold", fontsize=15); ax.set_ylabel("")
    for t in ax.texts: t.set_fontweight("bold")

fig.suptitle("Figure 2: LatAm ADRs — Sector, Industry, and Geographic Composition",
             fontsize=16, fontweight="bold", y=0.97)
show_and_save(fig, "sector_industry_piecharts.png")

### 6.4 Performance & risk snapshot


In [ ]:
# === 6.4 Performance snapshot (Sharpe, MDD, Ulcer) ===
px_perf = prices[strict_tickers].dropna(how="all")
lr      = np.log(px_perf / px_perf.shift(1)).dropna(how="all")

mu  = lr.mean() * 252
vol = lr.std() * np.sqrt(252)
sh  = mu / vol.replace(0, np.nan)
cum = (np.exp(lr.cumsum()).iloc[-1] - 1) * 100

def _ddstats(s):
    s = s.dropna()
    eq = np.exp(s.cumsum())
    dd = eq / eq.cummax() - 1
    return pd.Series({"mdd": dd.min()*100, "ui": np.sqrt((dd**2).mean())*100, "obs": len(s)})

dd_df = lr.apply(_ddstats).T
perf  = pd.DataFrame({
    "Sharpe": sh, "Ann. Ret %": mu*100, "Ann. Vol %": vol*100, "CumRet %": cum,
    "MDD %": dd_df["mdd"], "UlcerIndex": dd_df["ui"], "Days": dd_df["obs"].astype(int),
    "Sector": funda_liq["Sector"],
})
tbl = pd.concat([perf.sort_values("Sharpe").head(10), perf.sort_values("Sharpe").tail(10)])

def _c(v):
    if pd.isna(v):   return ""
    if v >=  0.5:    return "background-color:#1f7a1f;color:white"
    if v >=  0.3:    return "background-color:#4CAF50;color:white"
    if v <= -0.2:    return "background-color:#8B0000;color:white"
    if v <= -0.1:    return "background-color:#B22222;color:white"
    return                  "background-color:#8B8000;color:white"

display(Markdown("#### **Figure 3: Performance Snapshot — Bottom & Top 10 by Sharpe**"))
display(tbl.style.map(_c, subset="Sharpe").format({
    "Sharpe":"{:.2f}","Ann. Ret %":"{:.1f}","Ann. Vol %":"{:.1f}","CumRet %":"{:.1f}",
    "MDD %":"{:.1f}","UlcerIndex":"{:.2f}","Days":"{:,d}"}))

### 6.5 Clustered correlation heatmap + candidate pairs


In [ ]:
# === 6.5 Clustered correlation heatmap ===
LOOKBACK = CONFIG["CORR_LOOKBACK_DAYS"]
TH       = CONFIG["CORR_THRESHOLD"]

px_c = prices[strict_tickers].iloc[-LOOKBACK:].dropna(axis=1, how="any")
lr_c = np.log(px_c).diff().dropna()
corr = lr_c.corr()

Z     = linkage(squareform(1 - corr, checks=False), method="average")
order = corr.index[leaves_list(Z)]
C     = corr.loc[order, order]

hmask = np.triu(np.ones_like(C, bool))
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(C, mask=hmask, cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.2,
            linecolor="gray", cbar_kws={"label":"ρ (corr)"}, ax=ax)
ax.set_title(f"Figure 4: Clustered Correlation ({LOOKBACK}d, {len(C)} ADRs)",
             fontsize=14, fontweight="bold")
ax.tick_params(axis="x", rotation=90)
show_and_save(fig, "corr_heatmap_clustered.png")

cand = C.where(~hmask & ~np.eye(len(C), dtype=bool)).stack()
hp = (cand[cand.abs() >= TH].rename("Correlation")
      .rename_axis(["Ticker1", "Ticker2"]).reset_index())
display(Markdown(f"### **Candidate Pairs (|ρ| ≥ {TH})**"))
display(hp.sort_values("Correlation", key=lambda x: x.abs(), ascending=False))

### 6.6 Correlation network


In [ ]:
# === 6.6 Correlation network (|ρ| ≥ TH) ===
nmask = np.triu(np.ones_like(corr, bool), 1)
npairs = corr.where(nmask).stack()
edges  = [(i, j, {"weight": abs(w)}) for (i, j), w in npairs.items() if abs(w) >= TH]
G = nx.Graph(); G.add_edges_from(edges)
print(f"[INFO] nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()} (|ρ| ≥ {TH})")
pos = nx.spring_layout(G, seed=42, k=0.35)
fig, ax = plt.subplots(figsize=(6.5, 4))
nx.draw(G, pos, ax=ax, with_labels=True, node_color="#76c7d8", node_size=750,
        edge_color="#555", width=[2*d["weight"] for *_, d in G.edges(data=True)],
        font_size=8.5, font_weight="bold", font_color="#222")
ax.set_title(f"Figure 5: Correlation Network (|ρ| ≥ {TH})", fontsize=11, fontweight="bold")
ax.set_axis_off(); plt.tight_layout(pad=0.5)
show_and_save(fig, "corr_network_compact.png")

### 6.7 Correlation dendrogram


In [ ]:
# === 6.7 Correlation dendrogram ===
px_d = prices[strict_tickers].iloc[-LOOKBACK:].dropna(axis=1, how="any")
px_d.columns = [str(c) for c in px_d.columns]
if px_d.shape[1] < 3:
    print("[WARN] not enough tickers for dendrogram.")
else:
    lr_d = np.log(px_d).diff().dropna()
    corr_d = lr_d.corr().fillna(0)
    Zd = linkage(squareform(1 - corr_d.values, checks=False), method="average")
    fig, ax = plt.subplots(figsize=(18, 6))
    dendrogram(Zd, labels=list(corr_d.index), leaf_rotation=90, leaf_font_size=8,
               color_threshold=0.7, ax=ax)
    ax.axhline(1 - TH, color="#888", ls="--", lw=1, alpha=0.7)
    ax.set_title(f"Figure 6: Correlation Dendrogram (cut line = ρ ≥ {TH})",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("Distance (1 - ρ)"); plt.tight_layout()
    show_and_save(fig, "corr_dendrogram.png")

## 7. Export for NB 02

Writes two universes under stable filenames NB02 consumes directly:

- **Strict / institutional** (`tickers_institutional.csv`,
  `universe_institutional_fundamentals.csv`) — the §6.1 screen, for display.
- **Loose / pairs** (`tickers_pairs.csv`, `universe_pairs_fundamentals.csv`,
  `prices_pairs.csv`, `returns_pairs.csv`) — looser size/liquidity tier +
  the trailing-window coverage filter (`PAIRS_COVERAGE_*`), which is what
  NB 02's OU engine consumes. Tickers without ≥60% coverage over the last
  756 trading days drop here — this is where VTRU (delisted Jun-2024) exits.

`prices`/`returns` for the pairs export use distinct local names so the full
§4 panels in memory aren't clobbered.

In [ ]:
# === 7. Export universes for NB 02 ===
snapshot = CONFIG["SNAPSHOT_DATE"].strftime("%Y-%m-%d")

# Strict / institutional tier (funda_liq already built by §6.1)
manifest_strict = persist_universe_tier(funda_liq, tier="strict", snapshot=snapshot, paths=PATHS)

# Loose / pairs tier: size + liquidity threshold, then ≥60% coverage over the last 756 days
mask_loose = ((fundamentals["MarketCapitalization"] > CONFIG["MCAP_MIN_LOOSE"]) &
              ((fundamentals["ADV_63"]          > CONFIG["ADV_MIN_LOOSE"]) |
               (fundamentals["DollarVolume_3M"] > CONFIG["DOLLAR_MIN_LOOSE"])))
funda_loose   = fundamentals[mask_loose]
loose_tickers = sorted(set(funda_loose.index) & set(prices.columns))
keep          = prices[loose_tickers].iloc[-CONFIG["PAIRS_COVERAGE_DAYS"]:].notna().mean().ge(CONFIG["PAIRS_COVERAGE_MIN"])
final_tickers = keep[keep].index.tolist()
cov_dropped   = sorted(set(loose_tickers) - set(final_tickers))
if cov_dropped:
    print(f"[INFO] dropped by coverage: {cov_dropped}")
assert not (set(final_tickers) & EXCLUDE), "EXCLUDE leaked into pairs universe"

pairs_px   = prices[final_tickers]
pairs_rets = np.log(pairs_px).diff()
manifest_loose = persist_universe_tier(
    funda_loose.loc[final_tickers], pairs_px, pairs_rets,
    tier="loose", snapshot=snapshot, paths=PATHS,
)

print("[OK] NB01 universe artifacts written for NB02:")
for tier, m in [("strict", manifest_strict), ("loose", manifest_loose)]:
    for name, path in m.items():
        print(f"  [{tier}] {name}: {path.relative_to(PATHS.ROOT)}")
print(f"[INFO] in pairs but not strict: {sorted(set(final_tickers) - set(strict_tickers))}")
print("[INFO] pairs by country: " +
      ", ".join(f"{c}={n}" for c, n in
                funda_loose.loc[final_tickers, "CountryOfOrigin"].value_counts().items()))

---

**NB 01 complete.** Universe + fundamentals + price/volume/returns panels + earnings calendar sourced live from EODHD; diagnostic layer rendered; export contract for NB 02 satisfied.